# Notebook note

This notebook is where I work through the mean-field dropout criticality calculations. It contains the fixed-point helpers, recursion simulations, and linearized checks I use to compare the full map against the local asymptotics. If you only want the paper figures, run `python scripts/make_figures.py --all`; this notebook is for changing the recursions, checking fit windows, and keeping the theory honest.


# Mean-field dropout criticality

In previous literature, it was shown that the c^*=1 correlation fixed point was destroyed when adding independent dropout masks. A new fixed point was not calculated, yet from a physical point of view it exists. Finding it then allows us to define the correlation length as the lengthscale for the exponential decay (a half life of information decay of sorts) even for networks with dropout. Given that the recursion relations can be worked out as I do in the paper, we can use numerical solvers to work out what the fixed points will be.
 We use tanh and ReLU activations here as they are simple enough to compute and also illustrative since they belong to two distinct universality classes. While these are not the SOTA activations, they are common enough that they serve a pedagogical purpose as they are also slightly simpler to solve for analytically, and the universality classes ensure that the qualitative outcomes can be transferred to other modern architectures lying in either of the classes.

The quantities I keep tracking are:

- the correlation length $\xi$, which tells us how slowly correlations relax near criticality;
- the fixed-point gap $m_* = 1-c_*$, which is the order parameter for decorrelation;
- the critical relaxation curve $m_\ell = 1-c_\ell$ at exactly $\chi=1$;
- the dropout field $h$, which is the clean mean-field way to talk about dropout pushing $F(1)$ below one.


In [ ]:
from dropout_mft.paths import project_root

REPO_ROOT = project_root()

import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermgauss
from dropout_mft.numerics import find_positive_root, solve_bracketed
import warnings
warnings.filterwarnings("ignore")

# Basic numerical settings
GH_N = 60  # Gauss-Hermite quadrature order
KAPPA = 2 * np.sqrt(2) / (3 * np.pi)  # ReLU kink coefficient

# Plot style for the quick diagnostics
plt.rcParams.update({
    "font.family": "serif", "font.size": 11, "axes.labelsize": 13,
    "axes.titlesize": 13, "legend.fontsize": 9, "figure.dpi": 150,
    "text.usetex": False, "mathtext.fontset": "cm", "axes.linewidth": 0.8,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": "--",
})

from dropout_mft.style import COLORS, FIELD_PALETTE

PALETTE = FIELD_PALETTE

print(f"Using GH_N = {GH_N} (Gauss-Hermite quadrature)")
print(f"ReLU kink coefficient κ = {KAPPA:.8f}")

## Core numerical tools

Here I set up the numerical tools that the rest of the notebook keeps using. The Gauss-Hermite nodes are built once so that every Gaussian expectation is computed in the same deterministic way, and the one- and two-dimensional expectation helpers are just wrappers around those weights.

The fixed point equation is written in terms of $m=1-c$ because the interesting regime is close to $c=1$, and working directly in $m$ makes the small numbers easier to control. The actual root solving is still done by scipy; the only extra logic is finding the positive bracket that corresponds to the physical fixed point. The last helper is the log-log fit, which is how I extract the scaling exponents from the numerical curves.


In [ ]:
# Build Gauss-Hermite quadrature nodes once and reuse them throughout.
_x, _w = hermgauss(GH_N)
_z = np.sqrt(2.0) * _x
_w1 = _w / np.sqrt(np.pi)
_Z1, _Z2 = _z[:, None], _z[None, :]
_W2 = (_w[:, None] * _w[None, :]) / np.pi

# Shared expectation helpers used by the calculations below.
def gh_expect_1d(fvals):
    """E[f(z)] for z ~ N(0,1)."""
    return float(np.sum(_w1 * fvals))

def gh_expect_2d(fvals):
    """E[f(z1,z2)] for iid z1, z2 ~ N(0,1)."""
    return float(np.sum(_W2 * fvals))

# Tanh activation and derivatives; these are analytic and cheap to evaluate.
def tanh_phi(x): return np.tanh(x)
def tanh_phi_deriv(x): return 1.0 / np.cosh(x)**2
def tanh_phi_deriv2(x): return -2.0 * np.tanh(x) * tanh_phi_deriv(x)

# Root solving lives in dropout_mft.numerics, which wraps scipy.optimize.root_scalar.

# Log-log fits used to extract the scaling exponents.
def loglog_fit(x, y, fit_slice=slice(None)):
    """Fit ln(y) = a ln(x) + b and return slope/intercept standard errors."""
    x, y = np.asarray(x)[fit_slice], np.asarray(y)[fit_slice]
    mask = (x > 0) & (y > 0) & np.isfinite(x) & np.isfinite(y)
    coeffs, cov = np.polyfit(np.log(x[mask]), np.log(y[mask]), 1, cov=True)
    return float(coeffs[0]), float(coeffs[1]), float(np.sqrt(cov[0,0])), float(np.sqrt(cov[1,1]))

def power_law(x, a, b):
    """Evaluate the fitted law y = exp(b) x^a."""
    return np.exp(b) * np.asarray(x)**a

def style_log_axes(ax, legend_loc=None):
    ax.grid(True, which='major', alpha=0.30, ls='-', lw=0.8, color=COLORS['grid'])
    ax.grid(True, which='minor', alpha=0.15, ls=':', lw=0.5, color=COLORS['grid'])
    ax.set_axisbelow(True)
    if legend_loc:
        ax.legend(loc=legend_loc, frameon=True, fancybox=True, shadow=True, framealpha=0.95)

### Tanh mean-field functions

This is the smooth activation case. For tanh the correlation map can be evaluated once the Gaussian integrals are known, but first we need to find the variance fixed point $q_*$. Once that is solved, we can evaluate the correlation map $F(c)$, its derivative, and the stable fixed point $c_*$.

The important quantities are the same as in the paper. $\chi=F'(1)$ tells us where the correlation edge is when dropout is absent, $h=1-F(1)$ measures the way dropout moves the perfectly correlated point, and $g$ is the quadratic coefficient that makes this the smooth universality class.


In [ ]:
# Smooth activation utilities. I keep these close to the equations:
# solve q*, evaluate the map, differentiate it, then solve F(c)=c in m=1-c.
def solve_qstar_tanh(sigma_w2, sigma_b2, rho, tol=1e-13, max_iter=5000):
    """Solve variance fixed point: q = (σ²_w/ρ) E[tanh(√q z)²] + σ²_b"""
    q = sigma_b2 + 1.0
    for _ in range(max_iter):
        Ef2 = gh_expect_1d(tanh_phi(np.sqrt(max(q, 0)) * _z)**2)
        q_new = (sigma_w2 / rho) * Ef2 + sigma_b2
        if abs(q_new - q) < tol * max(1.0, q_new):
            return float(q_new)
        q = 0.5 * (q + q_new)
    return float(q)

def tanh_chi_g_h(sigma_w2, sigma_b2, rho):
    """Return (q*, χ, g, h) where χ=F'(1), g=F''(1), h=1-F(1)."""
    q = solve_qstar_tanh(sigma_w2, sigma_b2, rho)
    sq = np.sqrt(q)
    Ef2 = gh_expect_1d(tanh_phi(sq * _z)**2)
    Efp2 = gh_expect_1d(tanh_phi_deriv(sq * _z)**2)
    Efpp2 = gh_expect_1d(tanh_phi_deriv2(sq * _z)**2)
    chi, g = sigma_w2 * Efp2, sigma_w2 * q * Efpp2
    h = 1.0 - (sigma_w2 * Ef2 + sigma_b2) / q
    return q, chi, g, h

def F_tanh(c, sigma_w2, sigma_b2, rho, qstar=None):
    """Correlation map: F(c) = (σ²_w E[φ(u₁)φ(u₂)] + σ²_b) / q*"""
    if qstar is None:
        qstar = solve_qstar_tanh(sigma_w2, sigma_b2, rho)
    c = float(np.clip(c, -1 + 1e-15, 1 - 1e-15))
    s, sq = np.sqrt(max(0, 1 - c*c)), np.sqrt(qstar)
    u1, u2 = sq * _Z1, sq * (c * _Z1 + s * _Z2)
    return float((sigma_w2 * gh_expect_2d(tanh_phi(u1) * tanh_phi(u2)) + sigma_b2) / qstar)

def Fp_tanh(c, sigma_w2, sigma_b2, rho, qstar=None):
    """Derivative: F'(c) = σ²_w E[φ'(u₁)φ'(u₂)]"""
    if qstar is None:
        qstar = solve_qstar_tanh(sigma_w2, sigma_b2, rho)
    c = float(np.clip(c, -1 + 1e-15, 1 - 1e-15))
    s, sq = np.sqrt(max(0, 1 - c*c)), np.sqrt(qstar)
    u1, u2 = sq * _Z1, sq * (c * _Z1 + s * _Z2)
    return float(sigma_w2 * gh_expect_2d(tanh_phi_deriv(u1) * tanh_phi_deriv(u2)))

def cstar_tanh(sigma_w2, sigma_b2, rho):
    """Solve F(c) = c for the stable fixed point."""
    q, chi, g, h = tanh_chi_g_h(sigma_w2, sigma_b2, rho)
    if abs(h) < 1e-15 and chi - 1 <= 0:
        return 1.0
    def g_m(m): return F_tanh(1-m, sigma_w2, sigma_b2, rho, qstar=q) - (1-m)
    disc = (chi-1)**2 + 2*g*h
    m0 = float(np.clip((chi-1 + np.sqrt(max(disc, 0))) / max(g, 1e-16), 1e-8, 1.0))
    return 1.0 - float(find_positive_root(g_m, m0))

def tune_sigw2_for_chi_tanh(target_chi, sigma_b2, rho):
    def f(sw2): return tanh_chi_g_h(sw2, sigma_b2, rho)[1] - target_chi
    return solve_bracketed(f, (0.05, 20.0))

### ReLU mean-field functions

This is the kinked activation case. The ReLU kernel is nicer to evaluate because it has a closed form, but the price is that the expansion near $c=1$ is not smooth in the same way as tanh. That is exactly why the powers change.

I parameterize the map by $\chi$ and $\rho$ directly here, and then solve the same fixed point equation in $m=1-c$. This keeps the ReLU code parallel to the tanh code above, even though the actual kernel is different.


In [ ]:
# Kinked activation utilities. The ReLU kernel is closed form, so this block
# is mostly the kernel, its derivative, and the same stable fixed-point solve.
def F_relu_base(c):
    """Arc-cosine kernel (order 1) for ReLU."""
    c = float(np.clip(c, -1, 1))
    return float((np.sqrt(max(0, 1 - c*c)) + (np.pi - np.arccos(c)) * c) / np.pi)

def F_relu(c, chi, rho):
    """ReLU correlation map with tunable χ and dropout ρ."""
    return float(chi * F_relu_base(c) + 1.0 - chi / rho)

def Fp_relu(c, chi, rho):
    """Derivative: F'(c) = χ (1 - arccos(c)/π)"""
    c = float(np.clip(c, -1 + 1e-15, 1 - 1e-15))
    return float(chi * (1.0 - np.arccos(c) / np.pi))

def cstar_relu(chi, rho):
    """Solve F(c) = c for the stable fixed point."""
    h, t = 1.0 - F_relu(1.0, chi, rho), chi - 1.0
    if abs(h) < 1e-15 and t <= 0:
        return 1.0
    def g_m(m): return F_relu(1-m, chi, rho) - (1-m)
    m0_h = (h / (max(chi, 1e-12) * KAPPA))**(2/3) if h > 0 else 0.0
    m0_t = (t / (max(chi, 1e-12) * KAPPA))**2 if t > 0 else 0.0
    m0 = float(np.clip(max(m0_h, m0_t, 1e-8), 1e-8, 1.0))
    return 1.0 - float(find_positive_root(g_m, m0))

## Part I: critical exponents without dropout

We sanity check the no-dropout scaling exponents before doing the dropout-dependent ones.

The cell below looks at three numerical signatures of the edge. First, the correlation length diverges as $\xi \sim |\chi-1|^{-\nu}$ when we approach from the ordered side. Second, the fixed-point gap behaves like $m_* \sim (\chi-1)^\beta$ on the chaotic side. Third, exactly at $\chi=1$, the relaxation curve decays like $m_\ell \sim \ell^{-p}$.

We use the same log-log helper function to fit and plot these powers.


In [ ]:
sigma_b2 = 0.1  # Fixed bias variance

# Locate the tanh critical weight variance by solving chi(sigma_w^2)=1.
sigw2_c = solve_bracketed(lambda sw2: tanh_chi_g_h(sw2, sigma_b2, 1.0)[1] - 1.0, (0.1, 10.0))
q_c, chi_c, g_c, h_c = tanh_chi_g_h(sigw2_c, sigma_b2, 1.0)
print(f"Critical σ²_w = {sigw2_c:.8f}, χ = {chi_c:.8f}, q* = {q_c:.6f}")

# (a) Correlation length nu: approach the edge from the ordered side, chi<1.
t_below = -exp_grid(1e-2, 1e-1, 20)
xi = -1.0 / np.log(1.0 + t_below)
a_xi, b_xi, se_nu, _ = loglog_fit(np.abs(t_below), xi)
nu = -a_xi

# (b) Order parameter beta: approach from the chaotic side, chi>1.
sigw2_grid = sigw2_c * (1 + exp_grid(1e-2, 1e-1, 20))
chi_t = np.array([tanh_chi_g_h(sw2, sigma_b2, 1.0)[1] for sw2 in sigw2_grid if tanh_chi_g_h(sw2, sigma_b2, 1.0)[1] > 1])
m_t = np.array([1 - cstar_tanh(sw2, sigma_b2, 1.0) for sw2 in sigw2_grid if tanh_chi_g_h(sw2, sigma_b2, 1.0)[1] > 1])
t_t = chi_t - 1

chi_r = 1 + exp_grid(1e-2, 1e-1, 20)
m_r = np.array([1 - cstar_relu(chi, 1.0) for chi in chi_r])
t_r = chi_r - 1

# Fit the smallest offsets first, since those are the asymptotic points.
fit_slice = slice(0, max(6, len(t_t)//2))
a_bt, b_bt, se_bt, _ = loglog_fit(t_t, m_t, fit_slice)
a_br, b_br, se_br, _ = loglog_fit(t_r, m_r, fit_slice)

# (c) Critical relaxation p: iterate the map at chi=1 and fit the late-time tail.
Lmax = 20000
c = 0.0
m_l_tanh = np.empty(Lmax)
for ell in range(Lmax):
    c = F_tanh(c, sigw2_c, sigma_b2, 1.0, q_c)
    m_l_tanh[ell] = 1 - c

c = 0.0
m_l_relu = np.empty(Lmax)
for ell in range(Lmax):
    c = F_relu_base(c)
    m_l_relu[ell] = 1 - c

ells = np.arange(1, Lmax + 1)
fit_win = (ells > 500) & (ells < 8000)
a_pt, b_pt, se_pt, _ = loglog_fit(ells[fit_win], m_l_tanh[fit_win])
a_pr, b_pr, se_pr, _ = loglog_fit(ells[fit_win], m_l_relu[fit_win])

print("\n" + "="*60)
print("Part I: Critical exponents (no dropout)")
print("="*60)
print(f"  ν (correlation length):  {nu:.4f} ± {se_nu:.4f}")
print(f"  β (tanh order param):    {a_bt:.4f} ± {se_bt:.4f}")
print(f"  β (ReLU order param):    {a_br:.4f} ± {se_br:.4f}")
print(f"  p (tanh relaxation):     {-a_pt:.4f} ± {se_pt:.4f}")
print(f"  p (ReLU relaxation):     {-a_pr:.4f} ± {se_pr:.4f}")

## Part II: dropout perturbation exponents

Now dropout is turned on while staying at the correlation edge. In the mean-field recursion, independent dropout masks create a field $h=1-F_\rho(1)$, so even perfectly correlated inputs no longer map exactly back to perfect correlation.

I check the two scaling laws $m_* \sim h^{1/\delta}$ and $\xi \sim h^{-\nu_\rho}$.


In [ ]:
p_list = np.concatenate([exp_grid(1e-3, 1e-2, 8), exp_grid(1e-2, 0.2, 15)])
rho_list = np.clip(1 - p_list, 1e-6, 1 - 1e-12)

# Smooth tanh at chi=1. For each dropout keep probability rho, retune sigma_w^2 so chi stays pinned at one.
m_s, xi_s, h_s = [], [], []
for rho in rho_list:
    sw2 = tune_sigw2_for_chi_tanh(1.0, sigma_b2, rho)
    q, chi, g, h = tanh_chi_g_h(sw2, sigma_b2, rho)
    cstar = cstar_tanh(sw2, sigma_b2, rho)
    lam = Fp_tanh(cstar, sw2, sigma_b2, rho, q)
    m_s.append(1 - cstar)
    xi_s.append(-1 / np.log(lam))
    h_s.append(h)
m_s, xi_s, h_s = np.array(m_s), np.array(xi_s), np.array(h_s)

# Kinked ReLU at chi=1. The closed-form kernel lets us compute h, c*, and lambda directly.
m_k, xi_k, h_k = [], [], []
for rho in rho_list:
    h = 1 - F_relu(1.0, 1.0, rho)
    cstar = cstar_relu(1.0, rho)
    lam = Fp_relu(cstar, 1.0, rho)
    m_k.append(1 - cstar)
    xi_k.append(-1 / np.log(lam))
    h_k.append(h)
m_k, xi_k, h_k = np.array(m_k), np.array(xi_k), np.array(h_k)

# Fit only the smallest-h decade, where the field-theory powers should be visible.
h_fit_max = 10 * min(h_s.min(), h_k.min())
fit_s, fit_k = h_s <= h_fit_max, h_k <= h_fit_max

a_ms, b_ms, se_ms, _ = loglog_fit(h_s[fit_s], m_s[fit_s])
a_mk, b_mk, se_mk, _ = loglog_fit(h_k[fit_k], m_k[fit_k])
a_xs, b_xs, se_xs, _ = loglog_fit(h_s[fit_s], xi_s[fit_s])
a_xk, b_xk, se_xk, _ = loglog_fit(h_k[fit_k], xi_k[fit_k])

print("\n" + "="*60)
print("Part II: Dropout perturbation exponents (small h fits)")
print("="*60)
print(f"  Smooth (tanh):")
print(f"    1/δ = {a_ms:.4f} ± {se_ms:.4f}  (δ = {1/a_ms:.3f} ± {se_ms/a_ms**2:.3f})")
print(f"    ν_ρ = {-a_xs:.4f} ± {se_xs:.4f}")
print(f"  Kinked (ReLU):")
print(f"    1/δ = {a_mk:.4f} ± {se_mk:.4f}  (δ = {1/a_mk:.3f} ± {se_mk/a_mk**2:.3f})")
print(f"    ν_ρ = {-a_xk:.4f} ± {se_xk:.4f}")

## Combined figure: critical exponents and dropout scaling

This cell collects the previous checks into one figure. The top row is the no-dropout critical behaviour, and the bottom row shows the dropout-dependent critical exponents.

Putting them in the same chart makes the different scaling behaviours clearer and emphasizes the different power laws of the two universality classes.


In [ ]:
fig, axes = plt.subplot_mosaic(
    [
        ['xi', 'xi', 'order', 'order', 'relax', 'relax'],
        ['field_m', 'field_m', 'field_m', 'field_xi', 'field_xi', 'field_xi'],
    ],
    figsize=(22, 12),
    facecolor='white',
    gridspec_kw=dict(height_ratios=[1, 1], hspace=0.35, wspace=0.35),
)
ax1, ax2, ax3 = axes['xi'], axes['order'], axes['relax']
ax4, ax5 = axes['field_m'], axes['field_xi']

POINT_KW = dict(ms=6, mew=0.5, mec='white')
FIT_KW = dict(ls='--', lw=2.5)

def log_points(ax, x, y, marker, color, label=None):
    ax.loglog(x, y, marker, color=color, label=label, **POINT_KW)

def log_fit(ax, x, a, b, color, label):
    ax.loglog(x, power_law(x, a, b), color=color, label=label, **FIT_KW)

def finish_log_panel(ax, xlabel, ylabel, legend_loc):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    style_log_axes(ax, legend_loc)

# (a) Correlation length: fit xi against distance to the edge from chi<1.
t_abs = np.abs(t_below)
log_points(ax1, t_abs, xi, 'o', COLORS['smooth'])
log_fit(ax1, t_abs, a_xi, b_xi, COLORS['fit'], fr"$\nu = {nu:.3f} \pm {se_nu:.3f}$")
finish_log_panel(ax1, r"$|t| = |\chi - 1|$", r"$\xi = -1/\ln(\chi)$", 'lower left')

# (b) Order parameter: compare smooth and kinked fixed-point gaps for chi>1.
log_points(ax2, t_t, m_t, 'o', COLORS['smooth'], 'Smooth tanh')
log_points(ax2, t_r, m_r, 's', COLORS['kink'], 'Kinked ReLU')
tline = exp_grid(t_t.min(), t_t.max(), 200)
log_fit(ax2, tline, a_bt, b_bt, COLORS['fit'], fr"$\beta_{{\mathrm{{tanh}}}} = {a_bt:.3f} \pm {se_bt:.3f}$")
log_fit(ax2, tline, a_br, b_br, COLORS['theory'], fr"$\beta_{{\mathrm{{ReLU}}}} = {a_br:.3f} \pm {se_br:.3f}$")
finish_log_panel(ax2, r"$t = \chi - 1$", r"$m_* = 1 - c_*$", 'upper left')

# (c) Critical relaxation: late-layer decay exactly at chi=1.
ax3.loglog(ells, m_l_tanh, color=COLORS['smooth'], lw=1.5, alpha=0.7, label='Smooth tanh')
ax3.loglog(ells, m_l_relu, color=COLORS['kink'], lw=1.5, alpha=0.7, label='Kinked ReLU')
log_fit(ax3, ells, a_pt, b_pt, COLORS['fit'], fr"$p_{{\mathrm{{tanh}}}} = {-a_pt:.3f} \pm {se_pt:.3f}$")
log_fit(ax3, ells, a_pr, b_pr, COLORS['theory'], fr"$p_{{\mathrm{{ReLU}}}} = {-a_pr:.3f} \pm {se_pr:.3f}$")
finish_log_panel(ax3, r"Layer $\ell$", r"$m_\ell = 1 - c_\ell$", 'upper right')

# (d) Dropout field response: m* as a power of h.
hline = exp_grid(min(h_k.min(), h_s.min()), max(h_k.max(), h_s.max()), 200)
log_points(ax4, h_s, m_s, 'o', COLORS['smooth'], 'Smooth tanh')
log_points(ax4, h_k, m_k, 's', COLORS['kink'], 'Kinked ReLU')
log_fit(ax4, hline, a_ms, b_ms, COLORS['fit'], fr"$1/\delta_{{\mathrm{{tanh}}}} = {a_ms:.3f} \pm {se_ms:.3f}$")
log_fit(ax4, hline, a_mk, b_mk, COLORS['theory'], fr"$1/\delta_{{\mathrm{{ReLU}}}} = {a_mk:.3f} \pm {se_mk:.3f}$")
finish_log_panel(ax4, r"Dropout field $h = 1-\bar{F}_\rho(1)$", r"$m_* = 1-c_*$", 'upper left')

# (e) Dropout field response: correlation length as a power of h.
log_points(ax5, h_s, xi_s, 'o', COLORS['smooth'], 'Smooth tanh')
log_points(ax5, h_k, xi_k, 's', COLORS['kink'], 'Kinked ReLU')
log_fit(ax5, hline, a_xs, b_xs, COLORS['fit'], fr"$\nu_{{\rho,\mathrm{{tanh}}}} = {-a_xs:.3f} \pm {se_xs:.3f}$")
log_fit(ax5, hline, a_xk, b_xk, COLORS['theory'], fr"$\nu_{{\rho,\mathrm{{ReLU}}}} = {-a_xk:.3f} \pm {se_xk:.3f}$")
finish_log_panel(ax5, r"Dropout field $h$", r"$\xi = -1/\ln\lambda$", 'upper right')

plt.savefig('combined_exponents_and_dropout_scaling.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## Part III: universal scaling collapse

In the paper, I show that dropout and detuning from criticality have similar effects on the reduction of the correlation function. After choosing the right variables, the equation of state is really an equation of two variables rather than three. This is the visual picture of the math. If the theory is capturing the right local recursion, curves with different $h$ should fall on top of each other after this rescaling.

For tanh, the smooth normal form gives

$$\tilde{m}=m\sqrt{g/(2h)}, \qquad \tilde{\tau}=\frac{1-\chi}{\sqrt{2gh}}.$$

For ReLU, the kinked normal form gives

$$\tilde{m}=\frac{m}{(h/\kappa)^{2/3}}, \qquad u=\frac{1-\chi}{\kappa^{2/3}h^{1/3}}.$$

The next cell generates the raw points, and the last cell plots raw coordinates next to collapsed coordinates so it is easy to see what the rescaling is doing.


In [ ]:
h_list = exp_grid(1e-4, 1e-1, 10)
t_vals = np.linspace(-0.3, 0.3, 50)

# Smooth tanh data: sweep field strength and chi-offset, then store the raw m, h, and g values needed for the collapse.
smooth_data = []
for h_proxy in h_list:
    rho = 1 / (1 + h_proxy)
    for t in t_vals:
        sw2 = tune_sigw2_for_chi_tanh(1 + t, sigma_b2, rho)
        q, chi, g, h = tanh_chi_g_h(sw2, sigma_b2, rho)
        m = 1 - cstar_tanh(sw2, sigma_b2, rho)
        smooth_data.append((h_proxy, t, h, g, m))
smooth = np.array(smooth_data)
h_proxy_s, t_s, h_s2, g_s, m_s2 = smooth.T
tau_tilde_s = -t_s / np.sqrt(2 * g_s * h_s2)
m_tilde_s = m_s2 * np.sqrt(g_s / (2 * h_s2))

# Kinked ReLU data: same sweep, using the ReLU closed-form map.
kink_data = []
for h_target in h_list:
    rho = 1 / (1 + h_target)
    for t in t_vals:
        m = 1 - cstar_relu(1 + t, rho)
        h = 1 - F_relu(1.0, 1 + t, rho)
        kink_data.append((h_target, t, h, m))
kink = np.array(kink_data)
h_target_k, t_k, h_k2, m_k2 = kink.T
u_k = -t_k / (KAPPA**(2/3) * h_k2**(1/3))
m_tilde_k = m_k2 / ((h_k2 / KAPPA)**(2/3))

# Theory curves: these are the universal normal-form predictions after rescaling.
tau_line = np.linspace(-6, 6, 600)
m_smooth_theory = np.sqrt(1 + tau_line**2) - tau_line

def kink_universal(u):
    out = np.empty_like(np.atleast_1d(u), dtype=float)
    for i, ui in enumerate(np.atleast_1d(u)):
        roots = np.roots([1, ui, 0, -1])
        real = roots[np.isclose(roots.imag, 0, atol=1e-10)].real
        real = real[real > 0]
        out[i] = real.max()**2 if len(real) else np.nan
    return out

u_line = np.linspace(-6, 6, 600)
m_kink_theory = kink_universal(u_line)

In [ ]:
# Plot raw coordinates beside collapsed coordinates. The raw panels show the
# field dependence; the collapsed panels test whether the rescaling removes it.
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
fig.patch.set_facecolor('white')

def plot_scatter(ax, h_vals, x, y, marker, t_arr, h_arr):
    for idx, hp in enumerate(h_list):
        mask = np.isclose(h_vals, hp)
        mask0 = mask & np.isclose(t_arr, 0.0, atol=1e-12)
        h0 = float(np.median(h_arr[mask0])) if mask0.any() else float(np.median(h_arr[mask]))
        ax.plot(x[mask], y[mask], marker, ms=5, color=PALETTE[idx],
               mew=0.5, mec='white', label=fr"$h\approx {h0:.0e}$", alpha=0.9)

# (a) Smooth raw: m as a function of t before rescaling.
plot_scatter(axs[0,0], h_proxy_s, t_s, m_s2, 'o', t_s, h_s2)
axs[0,0].set_xlabel(r"$t=\chi-1$"); axs[0,0].set_ylabel(r"$m=1-c_*$")
axs[0,0].legend(frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc='upper left')

# (b) Smooth collapse: same points in the smooth scaling variables.
plot_scatter(axs[0,1], h_proxy_s, tau_tilde_s, m_tilde_s, 'o', t_s, h_s2)
axs[0,1].plot(tau_line, m_smooth_theory, '-', color='#2D2D2D', lw=2.5, label="Theory", zorder=10)
axs[0,1].set_xlabel(r"$\tilde{\tau}=(1-\chi)/\sqrt{2gh}$"); axs[0,1].set_ylabel(r"$\tilde{m}=m\sqrt{g/(2h)}$")
axs[0,1].set_xlim(-1, 2); axs[0,1].set_ylim(0, 2)
axs[0,1].legend(frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc='upper right')

# (c) Kinked raw: ReLU points before rescaling.
plot_scatter(axs[1,0], h_target_k, t_k, m_k2, 's', t_k, h_k2)
axs[1,0].set_xlabel(r"$t=\chi-1$"); axs[1,0].set_ylabel(r"$m=1-c_*$")
axs[1,0].legend(frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc='upper left')

# (d) Kinked collapse: ReLU points in the kinked scaling variables.
plot_scatter(axs[1,1], h_target_k, u_k, m_tilde_k, 's', t_k, h_k2)
axs[1,1].plot(u_line, m_kink_theory, '-', color='#2D2D2D', lw=2.5, label="Theory", zorder=10)
axs[1,1].set_xlabel(r"$u=(1-\chi)/(\kappa^{2/3}h^{1/3})$"); axs[1,1].set_ylabel(r"$m/(h/\kappa)^{2/3}$")
axs[1,1].set_xlim(-1.25, 2); axs[1,1].set_ylim(0, 2)
axs[1,1].legend(frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc='upper right')

for ax in axs.flat:
    ax.grid(True, which='major', ls='-', alpha=0.3, lw=0.8, color=COLORS['grid'])
    ax.grid(True, which='minor', ls=':', alpha=0.15, lw=0.5, color=COLORS['grid'])
    ax.set_axisbelow(True)

plt.tight_layout(pad=1.5)
plt.savefig('scaling_collapse_improved.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()